# 10 · Final models and the Monday run
**Book chapters 21–24.** Train once on all data, then every Monday run the weekly pipeline:
fetch → forecast → watchdog → alert rules → ledger → bulletin → Telegram/email → dashboard files.

In [ ]:
# ▶ Run this cell first in every notebook (about 1 minute)
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = os.environ.get('LW_ROOT', '.')
CODE = f'{ROOT}/code/leptowatch'
os.environ['LW_DATA_ROOT'] = f'{ROOT}/data'
os.environ['PYTHONPATH'] = f'{CODE}/src'
sys.path.insert(0, f'{CODE}/src')
os.chdir(CODE)
!apt-get -qq install -y poppler-utils > /dev/null
%pip -q install -r requirements.txt
print('code:', CODE, '| data:', os.environ['LW_DATA_ROOT'])

In [ ]:
!python -m leptowatch.pipeline train-final

### Weekly run (offline first, to test without the web)

In [ ]:
os.environ['LW_DRY_RUN'] = '1'        # print messages instead of sending
!python -m leptowatch.pipeline weekly --offline

### The real Monday run (Earth Engine + new reports + Telegram)

In [ ]:
os.environ.pop('LW_DRY_RUN', None)
from google.colab import userdata          # store secrets in Colab's key icon, not in code
os.environ['LW_TELEGRAM_TOKEN'] = userdata.get('LW_TELEGRAM_TOKEN')
!python -m leptowatch.pipeline weekly

### Copy the files the dashboard needs, then push to GitHub (Streamlit Cloud redeploys)

In [ ]:
import shutil
from leptowatch.config import path
dst = f'{CODE}/dashboard/data'; os.makedirs(dst, exist_ok=True)
for p in [path('outputs','forecast_latest.json'), path('processed','oof_blend.parquet'),
          path('processed','alerts_backtest.parquet'), path('reports','results.json'),
          path('reports','ablation.json'), path('raw','boundaries','rdhs_26_simplified.geojson'),
          path('ledger','forecasts.jsonl')]:
    if p.exists(): shutil.copy(p, dst)
print(os.listdir(dst))

In [ ]:
# one-time: git init + remote. Store a GitHub fine-grained token in Colab secrets as GH_TOKEN.
GH_USER, REPO = 'your-github-name', 'leptowatch'
tok = userdata.get('GH_TOKEN')
!git config --global user.email "you@example.com" && git config --global user.name "RootBotZ"
!git add dashboard/data && git commit -m "weekly forecast" || true
!git push https://{tok}@github.com/{GH_USER}/{REPO}.git HEAD:main

### Preview the dashboard inside Colab

In [ ]:
!pip -q install streamlit pydeck
!nohup streamlit run dashboard/app.py --server.port 8501 > /tmp/st.log 2>&1 &
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))